# Example: Flux Balance Analysis of a Urea-Cycle Network

How do reaction directions and enzyme capacities constrain the rate of urea production? We will use a small metabolic network to connect the flux balance formulation from [the lecture](CHEME-5800-L6a-Lecture-FluxBalanceAnalysis-Fall-2026.ipynb) to a numerical solution.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * **Construct the reaction model:** Build the stoichiometric matrix and identify the internal reactions and exchanges that enter the steady-state balances.
> * **Set and interpret flux bounds:** Combine a thermodynamic direction heuristic with enzyme-capacity estimates, keeping the units and data limitations explicit.
> * **Solve and check the optimum:** Maximize urea export, verify the flux balances, and explain which capacity limits the nominal result.

In this example, we use a simplified urea-cycle network to calculate the maximum urea export rate and identify the enzyme capacity that limits production. Let's get started!

___

___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the packages and source files used here.

Let's set up our code environment:

In [ ]:
# Load packages, data paths, and lab functions -
include(joinpath(@__DIR__, "Include.jl")) # @__DIR__ locates this notebook's setup file

# Choose the plot appearance -
# After changing the theme, rerun this cell and the plotting cells.
# A single figure can override this choice with the keyword theme = :dark.
theme(:default)   # light background
# theme(:dark)   # dark background

See the [Julia documentation](https://docs.julialang.org/en/v1/) and the local [model](src/Types.jl) and [solver](src/Compute.jl) documentation for the types and calculations used here.

___

## Task 1: Build the FBA model

In this task, we build the stoichiometric matrix and initialize the flux balance model from [the reaction network](data/Network.net). The network contains four cycle reactions, a nitric oxide synthase branch, and fourteen exchanges involving eighteen metabolites. Thus, $\mathbf{S}\in\mathbb{R}^{18\times19}$ has one row per metabolite and one column per reaction.

<div>
  <center>
    <img
      src="figs/Fig-Urea-cycle-Schematic.png"
      alt="Schematic of the urea cycle"
        width="600"
    />
  </center>
</div>



Each exchange is written as `[] → metabolite`, where `[]` denotes the surroundings. Positive exchange flux means uptake; negative exchange flux means secretion. We will use this convention when setting the urea-export objective.

> __Model data:__
>
> * `S`, `species`, and `reactions` store the stoichiometric matrix and its row and column labels.
> * `fluxbounds` stores the lower and upper bounds in two columns.
> * `objective` stores one coefficient per reaction, initially zero.

The code returns `model` and `rd`, a dictionary of reaction equations for the flux table. Task 2 replaces the default bounds and sets the objective.

In [2]:
model, rd = let

    # Load the network -
    listofreactions = read_reaction_file(joinpath(_PATH_TO_DATA, "Network.net")); # reaction records
    S, species, reactions, rd = build_stoichiometric_matrix(listofreactions); # matrix and labels
    boundsarray = build_default_bounds_array(listofreactions); # bounds from reversibility flags

    # Build the model -
    model = build(MyPrimalFluxBalanceAnalysisCalculationModel, (
        S = S, # stoichiometric matrix
        fluxbounds = boundsarray, # lower and upper bounds; updated in Task 2
        species = species, # row labels for S
        reactions = reactions, # column labels for S
        objective = length(reactions) |> R -> zeros(R), # zero coefficients; set in Task 2
    ));

    # Return -
    model, rd
end;

___

## Task 2: Set the flux bounds and objective

In this task, we use thermodynamic estimates and enzyme capacities to set the flux bounds, then specify the urea-export objective.

> __Simplified enzyme-capacity bounds:__
>
> At the reference enzyme abundance, with saturating substrates and no allosteric regulation, the specific flux through enzyme-catalyzed reaction $j$ satisfies:
>
> $$
> -\delta_j V_{max,j}^{\circ}\leq\hat v_j\leq V_{max,j}^{\circ}.
> $$
>
> Here, $V_{max,j}^{\circ}$ is the nominal reaction capacity, with the same units as $\hat v_j$: $\mathrm{mmol\,gDW^{-1}\,h^{-1}}$. The reversibility parameter $\delta_j$ is zero for a forward-only reaction and one when both directions are allowed with equal capacities.

We first assign reaction directions, then calculate the capacities and update the model.

### Step 1: Assign reaction directions

The [recorded eQuilibrator estimates](data/urea_thermodynamics.csv) give standard transformed reaction Gibbs energies $\Delta_{\mathrm r}G_j^{\prime\circ}$ at pH 7.5, pMg 3.0, ionic strength 0.25 M, and 298.15 K. We assign directions using the heuristic:

$$
\delta_j=
\begin{cases}
1, & \Delta_{\mathrm r}G_j^{\prime\circ}>-10\,\mathrm{kJ\,mol^{-1}},\\
0, & \text{otherwise}.
\end{cases}
$$

This cutoff is a modeling assumption; physiological reversibility also depends on metabolite activities. Applying it to the recorded estimates gives:

<table style="display:table;width:100%;max-width:760px;table-layout:fixed;text-align:left;">
<colgroup><col style="width:12%;"><col style="width:44%;"><col style="width:32%;"><col style="width:12%;"></colgroup>
<thead><tr><th style="text-align:left;">Reaction</th><th style="text-align:left;">Enzyme</th><th style="text-align:right;">Δ<sub>r</sub><i>G</i><sub>j</sub><sup>′∘</sup> (kJ/mol)</th><th style="text-align:right;"><i>δ</i><sub>j</sub></th></tr></thead>
<tbody>
<tr><td>v1</td><td>Argininosuccinate synthetase</td><td style="text-align:right;">−4.3</td><td style="text-align:right;">1</td></tr>
<tr><td>v2</td><td>Argininosuccinate lyase</td><td style="text-align:right;">11.6</td><td style="text-align:right;">1</td></tr>
<tr><td>v3</td><td>Arginase</td><td style="text-align:right;">−33.9</td><td style="text-align:right;">0</td></tr>
<tr><td>v4</td><td>Ornithine transcarbamylase</td><td style="text-align:right;">−30.3</td><td style="text-align:right;">0</td></tr>
<tr><td>v5</td><td>Nitric oxide synthase</td><td style="text-align:right;">−1254.4</td><td style="text-align:right;">0</td></tr>
</tbody>
</table>

We store these assignments by reaction name in `reversibility_parameter_dictionary`. Exchange bounds are set separately in Step 3.

In [3]:
reversibility_parameter_dictionary = let
    ΔG_threshold = -10.0; # illustrative reversibility cutoff, kJ/mol
    data = CSV.read(joinpath(_PATH_TO_DATA, "urea_thermodynamics.csv"), DataFrame);
    Dict(row.reaction => Int(row.dg_prime_standard_kj_per_mol > ΔG_threshold)
         for row in eachrow(data));
end;

### Step 2: Estimate enzyme capacities

We combine the [recorded turnover numbers](data/urea_turnover_numbers.csv) with a common reference enzyme abundance $e^{\circ}=0.01\,\mathrm{mmol\,gDW^{-1}}$. These inputs set an illustrative capacity scale; they are not a matched set of HL-60 measurements.

The records give $k_{cat,j}^{\circ}$ in $\mathrm{s^{-1}}$. To express the reaction capacity in $\mathrm{mmol\,gDW^{-1}\,h^{-1}}$, we convert seconds to hours:

$$
V_{max,j}^{\circ}
=\underbrace{\left(3600\,\mathrm{s\,h^{-1}}\right)}_{\text{time conversion}}
 k_{cat,j}^{\circ}e^{\circ}.
$$

For the nominal inputs, the capacities are:

<table style="display:table;width:100%;max-width:880px;table-layout:fixed;text-align:left;">
<colgroup><col style="width:10%;"><col style="width:18%;"><col style="width:30%;"><col style="width:42%;"></colgroup>
<thead><tr><th style="text-align:left;">Reaction</th><th style="text-align:right;"><i>k</i><sub>cat,j</sub><sup>∘</sup> (s<sup>−1</sup>)</th><th style="text-align:right;"><i>V</i><sub>max,j</sub><sup>∘</sup> (mmol gDW<sup>−1</sup> h<sup>−1</sup>)</th><th style="text-align:left;">Turnover-number source</th></tr></thead>
<tbody>
<tr><td>v1</td><td style="text-align:right;">10.00</td><td style="text-align:right;">360.00</td><td>Default based on a global median</td></tr>
<tr><td>v2</td><td style="text-align:right;">3.28</td><td style="text-align:right;">118.08</td><td>Human/duck attribution conflict</td></tr>
<tr><td>v3</td><td style="text-align:right;">190.00</td><td style="text-align:right;">6840.00</td><td>Human arginase I</td></tr>
<tr><td>v4</td><td style="text-align:right;">410.00</td><td style="text-align:right;">14760.00</td><td><i>E. coli</i></td></tr>
<tr><td>v5</td><td style="text-align:right;">1.08</td><td style="text-align:right;">38.88</td><td>Rat nitric oxide synthase</td></tr>
</tbody>
</table>

The lyase reaction `v2` has the smallest capacity among the four cycle reactions. The smaller capacity of `v5` belongs to the competing branch. The data file records the assay conditions, source publications, and the species-attribution conflict for `v2`.

The code stores the capacities in `maximum_reaction_velocity_dictionary`. Changing the turnover numbers or enzyme abundance changes these values.

In [4]:
maximum_reaction_velocity_dictionary = let
    eₒ = 0.01; # illustrative reference enzyme abundance, mmol/gDW
    seconds_per_hour = 3600.0; # convert turnover numbers from s⁻¹ to h⁻¹
    data = CSV.read(joinpath(_PATH_TO_DATA, "urea_turnover_numbers.csv"), DataFrame);
    Dict(row.reaction => row[Symbol("model_kcat_s-1")]*eₒ*seconds_per_hour
         for row in eachrow(data));
end;

### Step 3: Update the flux bounds

The loop applies the enzyme-capacity bounds to `v1`–`v5`. Exchange reactions retain their bounds of $\pm1000\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$. These exchange limits do not bind at the nominal optimum, but they can constrain the solution if we increase the enzyme capacities.

In [5]:
fluxbounds = let
    
    fluxbounds = copy(model.fluxbounds); # edit a separate array before attaching it to the model
    names = model.reactions;
    for i ∈ eachindex(names)
        name = names[i]; # reaction label
    
        # Keep the exchange bounds -
        if startswith(name, "b") # b-prefixed labels identify exchanges in Network.net
            continue;
        end
        
        VMax = maximum_reaction_velocity_dictionary[name]; # capacity, mmol/gDW/h
        δᵢ = reversibility_parameter_dictionary[name]; # 0: forward-only; 1: reversible
    
        # Apply the enzyme-capacity bounds -
        fluxbounds[i,1] = -δᵢ*VMax; # allow reverse flux only when δᵢ = 1
        fluxbounds[i,2] = VMax; # forward capacity
    end

    # Return -
    fluxbounds;
end;

In [6]:
model.fluxbounds = fluxbounds;

Every flux interval includes zero, so the zero-flux vector satisfies both the bounds and the steady-state balances. We now set an objective that selects a solution with urea production.

### Step 4: Set the urea-export objective

The local [solver](src/Compute.jl) maximizes $\mathbf{c}^{\top}\hat{\mathbf{v}}$, where $\mathbf{c}$ contains the objective coefficients. Since `b4` is written as `[] → M_Urea_c`, secretion corresponds to a negative flux. Setting its coefficient to $-1$ and all others to zero gives the objective:

$$
\mathbf{c}^{\top}\hat{\mathbf{v}}=-\hat v_{b4}.
$$

Maximizing this quantity maximizes urea export. We clear any previous coefficients before setting the entry for `b4`, so rerunning the cell preserves the intended objective.

In [7]:
objective = model.objective; # shared array: entry changes update the model objective
fill!(objective, 0.0); # clear previous choices when this cell is rerun
reaction_to_maximize = "b4"; # urea exchange; export has negative flux
urea_exchange_index = findfirst(==(reaction_to_maximize), model.reactions);
@assert !isnothing(urea_exchange_index) "Urea exchange reaction b4 is missing";
objective[urea_exchange_index] = -1.0; # the solver maximizes -v_b4

___

## Task 3: Solve and interpret the flux balance model

In this task, we solve the linear program and examine the reaction fluxes and exchanges that support urea production.

[The solve(...) function](src/Compute.jl) returns the optimal flux vector in `solution["argmax"]` and the objective value in `solution["objective_value"]`. We report urea export as the negative of the signed `b4` flux. If the solver does not find a feasible optimum, the calculation stops with an error.


In [8]:
solution = solve(model); # stop here if the solver does not return a feasible optimum
urea_export_rate = -solution["argmax"][urea_exchange_index]; # convert uptake-positive flux to export rate, mmol/gDW/h
println("Maximum urea export: $(round(urea_export_rate; digits=2)) mmol/gDW/h");


Maximum urea export: 118.08 mmol/gDW/h


### Inspect the optimal fluxes

For the nominal parameters, the maximum urea export rate is $118.08\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$. The table shows each flux, its lower and upper bounds (`LB` and `UB`), and its reaction equation. All numerical columns use the same flux units; the results will change if we change the model parameters.


In [9]:
flux_table = let
    # Assemble the fluxes, bounds, and reaction equations -
    flux_bounds_array = model.fluxbounds;
    df = DataFrame(
        reaction = model.reactions,
        flux = solution["argmax"],
        LB = flux_bounds_array[:, 1],
        UB = flux_bounds_array[:, 2],
        equation = [rd[r] for r in model.reactions]
    );

    # Wrap long equations so the numerical columns remain visible -
    display(pretty_table(HTML, df;
        show_first_column_label_only = true,
        alignment = [:l, :r, :r, :r, :l],
        style = HtmlTableStyle(
            table = ["display" => "table", "width" => "100%", "table-layout" => "fixed"],
            first_line_column_label = [["font-weight" => "bold", "width" => width]
                for width in ["9%", "11%", "11%", "11%", "58%"]]
        ),
        highlighters = [HtmlHighlighter((data, i, j) -> j == 5,
            ["white-space" => "normal", "overflow-wrap" => "anywhere"])]
    ));
end


reaction,flux,LB,UB,equation
v1,118.08,-360.0,360.0,M_ATP_c+M_L-Citrulline_c+M_L-Aspartate_c = M_AMP_c+M_Diphosphate_c+M_N-(L-Arginino)succinate_c
v2,118.08,-118.08,118.08,M_N-(L-Arginino)succinate_c = M_Fumarate_c+M_L-Arginine_c
v3,118.08,0.0,6840.0,M_L-Arginine_c+M_H2O_c = M_L-Ornithine_c+M_Urea_c
v4,118.08,0.0,14760.0,M_Carbamoyl_phosphate_c+M_L-Ornithine_c = M_Orthophosphate_c+M_L-Citrulline_c
v5,0.0,0.0,38.88,2*M_L-Arginine_c+4*M_Oxygen_c+3*M_NADPH_c+3*M_H_c = 2*M_Nitric_oxide_c+2*M_L-Citrulline_c+3*M_NADP_c+4*M_H2O_c
b1,118.08,-1000.0,1000.0,[] = M_Carbamoyl_phosphate_c
b2,118.08,-1000.0,1000.0,[] = M_L-Aspartate_c
b3,-118.08,-1000.0,1000.0,[] = M_Fumarate_c
b4,-118.08,-1000.0,1000.0,[] = M_Urea_c
b5,118.08,-1000.0,1000.0,[] = M_ATP_c


The four cycle reactions carry equal fluxes, while the nitric oxide synthase branch `v5` is inactive. Positive exchanges supply carbamoyl phosphate, aspartate, ATP, and water. Negative exchanges remove fumarate, urea, AMP, diphosphate, and orthophosphate.

Reaction `v2` reaches its upper bound. To establish whether that bound limits urea production, we next examine how the internal balances couple the reaction fluxes.

### Why does the lyase capacity limit urea production?

A reaction at its upper bound does not necessarily limit the objective. Here, we can establish the connection using the balances for three internal metabolites:

$$
\begin{aligned}
\hat v_1-\hat v_2 &= 0 &&\text{(argininosuccinate)},\\
\hat v_3-\hat v_4 &= 0 &&\text{(ornithine)},\\
\hat v_2-\hat v_3-2\hat v_5 &= 0 &&\text{(arginine)}.
\end{aligned}
$$

Urea is produced by `v3` and exported through `b4`. Since the branch flux $\hat v_5$ is nonnegative, the export rate satisfies:

$$
-\hat v_{b4}=\hat v_3
=\hat v_2-2\hat v_5
\leq V_{max,2}^{\circ}.
$$

The nominal solution attains this bound with $\hat v_5=0$. Thus, the lyase capacity limits urea export to $118.08\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$. Doubling this capacity raises the optimum to $236.16\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$; the other nominal capacities and exchange bounds still allow that rate.


### What would oxygen uptake imply?

Only the nitric oxide synthase reaction `v5` consumes oxygen in this network. The oxygen balance therefore relates uptake through `b9` to the branch flux:

$$
\hat v_{b9}=4\hat v_5.
$$

Combining this relation with the arginine balance gives:

$$
-\hat v_{b4}=\hat v_2-\frac{1}{2}\hat v_{b9}.
$$

Positive oxygen uptake requires the branch to carry flux, diverting arginine from urea production. For example, fixing oxygen uptake at $4\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$ gives $\hat v_5=1\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$ and lowers the maximum urea export to $116.08\,\mathrm{mmol\,gDW^{-1}\,h^{-1}}$ with the nominal enzyme capacities. Whole-cell oxygen uptake cannot identify this branch by itself because other cellular reactions also consume oxygen.


### Check the numerical solution

The following checks confirm the model dimensions, finite flux values, steady-state mass balances, flux bounds, and agreement between the objective value and urea export. We allow a small numerical tolerance when comparing floating-point results. Passing these checks establishes consistency with the model constraints; it does not validate the biological assumptions.


In [10]:
let
    flux = solution["argmax"];
    tolerance = 1e-7; # absolute numerical tolerance, mmol/gDW/h
    @testset "Urea-cycle FBA solution checks" begin
        @test size(model.S) == (18, 19)
        @test all(isfinite, flux)
        @test maximum(abs.(model.S*flux)) <= tolerance
        @test all(flux .>= model.fluxbounds[:, 1] .- tolerance)
        @test all(flux .<= model.fluxbounds[:, 2] .+ tolerance)
        @test isapprox(solution["objective_value"], urea_export_rate; atol=tolerance)
        @test urea_export_rate >= -tolerance
    end
end;

Test Summary:                  | Pass  

Total  Time
Urea-cycle FBA solution checks |    7      7  0.5s


___

## Summary

In this example, we constructed a urea-cycle flux balance model, used thermodynamic and kinetic records to set its bounds, and interpreted the solution that maximizes urea export.

> __Key Takeaways:__
>
> * __Parameter choices define the feasible fluxes:__ We combined a thermodynamic direction rule with enzyme capacities in consistent units. The resulting bounds describe an illustrative network assembled from mixed-organism data.
> * __Network conventions determine the objective:__ We used uptake-positive exchange reactions, so maximizing urea export required a negative coefficient on the urea exchange flux.
> * __Balances explain the limiting capacity:__ We used the internal balances and a capacity perturbation to establish that argininosuccinate lyase limits nominal urea production. Oxygen uptake forces the competing branch to consume arginine and reduces the maximum export rate.

We can now use the model to test how changes in enzyme capacity and competing branch activity alter the maximum urea export rate.

___
